In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# LOADING THE LABELED DATA
try:
    df = pd.read_excel('labeled_final_dataset.xlsx')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: 'labeled_final_dataset.xlsx' not found.")
    exit()

# SEPARATING FEATURES (X) AND TARGET (y)
y = df['BPL_Target']

# Identifying direct income columns to drop (preventing Target Leakage)
# Wanting the model to learn based on jobs, vehicles, and assets, not direct income counts.
income_cols = [col for col in df.columns if col.startswith('in') and any(char.isdigit() for char in col)]

cols_to_drop = ['hasfamilyid', 'BPL_Target', 'familyRange', 'district', 'RuralUrban', 'isChild', 'isHousewife', 'isSenior Citizen', 'isStudent'] + income_cols 
# Comment '+ income_cols' and the lines of code above the comment in the 100-Families.ipynb to add the Per Member Salary to the training dataset
X = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

print(f"Total features selected for training: {X.shape[1]}")

# STRATIFYING TRAIN-TEST SPLIT
# 'stratify=y' guarantees that the imbalance ratio is preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)
print(f"Training set size: {X_train.shape[0]} families")
print(f"Testing set size: {X_test.shape[0]} families")

# BUILDING THE BALANCED PIPELINES
# We use StandardScaler for Logistic Regression and SVM, but exclude it for Naive Bayes and Random Forest as they perform better on unscaled binary/categorical data.
pipelines = {
    "Logistic Regression (Penalized)": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
    ]),
    "SVM (Linear Kernel)": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', SVC(kernel='linear', class_weight='balanced', random_state=42))
    ]),
    "Bernoulli Naive Bayes": Pipeline([
        ('classifier', BernoulliNB())
    ]),
    "Random Forest (Shallow Trees)": Pipeline([
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=3, class_weight='balanced', random_state=42))
    ])
}

# STRATIFIED CROSS-VALIDATION SETTINGS
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
leaderboard_results = []

# LOOPING THROUGH EACH ALGORITHM FOR EVALUATION
for name, pipeline in pipelines.items():
    print(f"\n  MODEL: {name.upper()}")
    
    # STRATIFIED CROSS-VALIDATION
    print("\n--- Running 5-Fold Stratified Cross-Validation ---")
    # This tests the model 5 times on different slices of the training data
    cv_results = cross_validate(pipeline, X_train, y_train, cv=cv_strategy, scoring=['f1_macro', 'accuracy'])
    
    mean_f1 = np.mean(cv_results['test_f1_macro'])
    std_f1 = np.std(cv_results['test_f1_macro'])
    print(f"Cross-Validation Macro F1-Score: {mean_f1:.4f} (± {std_f1:.4f})")
    if std_f1 > 0.1:
        print("Note: High standard deviation indicates the model's performance fluctuates depending on the data slice.")

    # FINAL TRAINING & EVALUATION ON UNSEEN TEST DATA
    print("\n--- Final Model Evaluation on Unseen Test Data ---")
    # Training the model on the full training set
    pipeline.fit(X_train, y_train)

    # Predicting on the unseen 20% testing set
    y_pred = pipeline.predict(X_test)
    
    # Calculating metrics for the leaderboard
    holdout_macro_f1 = f1_score(y_test, y_pred, average='macro')
    holdout_acc = accuracy_score(y_test, y_pred)
    
    leaderboard_results.append({
        "Algorithm": name,
        "CV Train F1": round(mean_f1, 4),
        "CV Variance": round(std_f1, 4),
        "Holdout F1": round(holdout_macro_f1, 4),
        "Holdout Acc": round(holdout_acc, 4)
    })

    # Displaying the classification report and confusion matrix
    print("\nClassification Report:")
    # Target names mapped to our labels
    print(classification_report(y_test, y_pred, target_names=['Non-BPL (0)', 'BPL (1)']))

    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(f"True Negatives (Correct Non-BPL): {cm[0][0]}")
    print(f"False Positives (Fraudulent BPL Guesses): {cm[0][1]}")
    print(f"False Negatives (Missed BPL Families): {cm[1][0]}")
    print(f"True Positives (Correct BPL): {cm[1][1]}")
    print("\n")

# PRINTING THE CONSOLIDATED LEADERBOARD
print("                 🏆 OVERALL ALGORITHM LEADERBOARD 🏆                 ")
# Sort by the F1-Score achieved on the 20% Unseen Holdout data
leaderboard_df = pd.DataFrame(leaderboard_results).sort_values(by="Holdout F1", ascending=False)
print(leaderboard_df.to_string(index=False))

Dataset loaded successfully.
Total features selected for training: 12
Training set size: 80 families
Testing set size: 20 families

  MODEL: LOGISTIC REGRESSION (PENALIZED)

--- Running 5-Fold Stratified Cross-Validation ---
Cross-Validation Macro F1-Score: 0.6861 (± 0.1050)
Note: High standard deviation indicates the model's performance fluctuates depending on the data slice.

--- Final Model Evaluation on Unseen Test Data ---

Classification Report:
              precision    recall  f1-score   support

 Non-BPL (0)       0.88      0.88      0.88         8
     BPL (1)       0.92      0.92      0.92        12

    accuracy                           0.90        20
   macro avg       0.90      0.90      0.90        20
weighted avg       0.90      0.90      0.90        20

Confusion Matrix:
True Negatives (Correct Non-BPL): 7
False Positives (Fraudulent BPL Guesses): 1
False Negatives (Missed BPL Families): 1
True Positives (Correct BPL): 11



  MODEL: SVM (LINEAR KERNEL)

--- Running 